In [ ]:
# config
samples = ["A1", "A2", "B2", "C2", "D1"]
input_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/raw_data"
cellpose_model_path = "/scratch/leuven/357/vsc35768/.cache/cellpose/cpsam"
plotting = True
on_hpc = True
unit_testing = False

In [ ]:
from pathlib import Path
import pandas as pd
import os
import gc
from datetime import date
from packaging import version

from dask_image import imread
from spatialdata import SpatialData
from spatialdata.transformations import set_transformation, Identity
from spatialdata.models import PointsModel
import harpy as hp
import cellpose
import torch
from harpy.image import cellpose_callable

In [ ]:
if on_hpc:
    # setting cache to scratch for torch and cellpose
    os.environ['TORCH_HOME'] = os.path.join(os.environ['VSC_SCRATCH'], '.cache/torch')
    os.environ['CELLPOSE_LOCAL_MODELS_PATH'] = os.path.join(os.environ['VSC_SCRATCH'], '.cache/cellpose')

    print(f"TORCH_HOME: {os.environ.get('TORCH_HOME')}")
    print(f"CELLPOSE_LOCAL_MODELS_PATH: {os.environ.get('CELLPOSE_LOCAL_MODELS_PATH')}")

    # check GPU availability in PyTorch
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    pass

## Loading the data

In [ ]:
def load_images_transcritps(folder: str, samples: list[str]) -> SpatialData:
    sdata = SpatialData()
    folder = Path(folder)
    for sample in samples:
        # load in the images
        matches = sorted(folder.glob(f"{sample}_DAPI.tiff"))
        if matches:
            p = matches[0]
            sdata = hp.im.add_image_layer(
                sdata,
                arr = imread.imread(str(p)),
                output_layer = f"{sample}_DAPI",
                transformations = {sample: Identity()},
                overwrite=True,
            )
        else:
            print(f"Warning: No DAPI image found for {sample}")
        

        # loading the transcripts
        df = pd.read_csv(
            Path(folder) / f"{sample}_results.txt",
            sep = r"\s+",
            header = None,
            names = ["x", "y", "z", "gene"],
            engine = "python",
        )
        points = PointsModel.parse(df, coordinates={"x": "x", "y": "y"})
        # set coordinate system for the transcripts
        points.attrs['transform'] = {} # ensuring no global coordinate system is set
        set_transformation(points, transformation = Identity(), to_coordinate_system = sample)
        sdata.points[f"{sample}_transcripts"] = points

    return sdata

In [ ]:
sdata = load_images_transcritps(
    folder = input_path,
    samples = samples
)
sdata

In [ ]:
sdata.images["A1_DAPI"]

## Image processing

In [ ]:
# min max filtering
for sample in samples:
    sdata = hp.im.min_max_filtering(
        sdata,
        img_layer = f"{sample}_DAPI",                 
        output_layer = f"{sample}_min_max_filtered", 
        size_min_max_filter = 46,
        overwrite = True,
    )

if plotting == True:
    for sample in samples:
        hp.pl.plot_image(
            sdata, 
            img_layer = [f"{sample}_DAPI", f"{sample}_min_max_filtered"], 
            crd = [4000, 8000, 6000, 8000], 
            figsize = (20,20),
            to_coordinate_system = sample
        )

In [ ]:
# enhance contrast
for sample in samples:
    sdata = hp.im.enhance_contrast(
        sdata,
        img_layer = f"{sample}_min_max_filtered",
        output_layer = f"{sample}_clahe",
        contrast_clip = 20,
        chunks = 20000,
        overwrite = True
    )

# Plot the contrast enhanced image
if plotting == True:
    for sample in samples:
        hp.pl.plot_image(
            sdata, 
            img_layer = [f"{sample}_min_max_filtered", f"{sample}_clahe"], 
            crd = [4000, 8000, 6000, 8000], 
            figsize = (20,20),
            to_coordinate_system = sample
        )

In [ ]:
sdata

## Cell segmentation

In [ ]:
# checking what is available on the system
cellpose_version = version.parse(cellpose.version)
if torch.backends.mps.is_available() and cellpose_version >= version.parse("4.0"):  # mps bugged in cellpose < 4.0
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}.")

In [ ]:
# this needs to be done on a GPU (took 8min on NVIDIA A100-SXM4-80GB for all samples)
for sample in samples:
    sdata = hp.im.segment(
        sdata,
        img_layer= f"{sample}_clahe",
        chunks = 4096,
        depth = 40,
        model = cellpose_callable,
        # parameters that will be passed to the callable _cellpose:
        pretrained_model = cellpose_model_path, 
        device = device,
        diameter = 180,
        flow_threshold = 0.9,
        cellprob_threshold = -6,
        min_size = 160,
        output_labels_layer = f"{sample}_segmentation_mask",
        output_shapes_layer = f"{sample}_segmentation_mask_boundaries",
        #crd=[6000, 10096, 6000, 10096] if unit_testing else None, 
        to_coordinate_system = sample,
        overwrite = True,
    )
    gc.collect() # freeing memory
    torch.cuda.empty_cache() # empty caching

In [ ]:
sdata

In [ ]:
if plotting == True:
    for sample in samples:
        hp.pl.plot_shapes(
            sdata, 
            img_layer = f"{sample}_clahe", 
            shapes_layer = f"{sample}_segmentation_mask_boundaries", 
            figsize=(10,10), 
            to_coordinate_system = f"{sample}",
            crd = [4000, 6000, 3000, 5000]
        )

## Transcript allocation

In [ ]:
for sample in samples:
    sdata = hp.tb.allocate(
        sdata = sdata,
        labels_layer = f"{sample}_segmentation_mask", 
        points_layer = f"{sample}_transcripts", 
        output_layer = f"{sample}_transcriptomics",
        update_shapes_layers = False,
        overwrite = True,
        to_coordinate_system = sample
    )

In [ ]:
# qc of transcript allocation
def transcript_allocation_qc(sdata, sample):
    print(f"For {sample}:")
    print("Number of transcripts in points layer of sample: ", len(sdata.points[f"{sample}_transcripts"]))
    print("Number of transcripts assigned to cells: ", sdata.tables[f"{sample}_transcriptomics"].X.sum())
    print("Percentage of transcripts allocated: ", ((sdata.tables[f"{sample}_transcriptomics"].X.sum())/len(sdata.points[f"{sample}_transcripts"]))*100)

In [ ]:
for sample in samples:
    transcript_allocation_qc(
        sdata = sdata,
        sample = sample
    )

In [ ]:
# write the object to zarr
sdata.write(f"/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260126_final.zarr", overwrite = True)